In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

2024-04-10 14:02:59.483824: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-04-10 14:03:00.369951: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-04-10 14:03:02.458407: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-10 14:03:04.421652: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
import re
import nltk
from nltk.util import pr
from nltk.stem import WordNetLemmatizer
#stemmer = nltk.SnowballStemmer("english")
from nltk.corpus import stopwords
import string 
#stopword = set(stopwords.words("english"))

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize NLTK components
stopword = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/codespace/nltk_data...


In [5]:
df = pd.read_csv("/workspaces/COMP710-001-Project-Cyberbullying-detection-System-with-front-end-integration/Dataset/twitter_parsed_dataset.csv")
print(df.head())
#print(df.isnull().sum())
# Handle NaN values
df.dropna(inplace=True)  # Drop rows with NaN values


                   index                     id  \
0  5.74948705591165E+017  5.74948705591165E+017   
1  5.71917888690393E+017  5.71917888690393E+017   
2  3.90255841338601E+017  3.90255841338601E+017   
3  5.68208850655916E+017  5.68208850655916E+017   
4  5.75596338802373E+017  5.75596338802373E+017   

                                                Text Annotation  oh_label  
0  @halalflaws @biebervalue @greenlinerzjm I read...       none       0.0  
1  @ShreyaBafna3 Now you idiots claim that people...       none       0.0  
2  RT @Mooseoftorment Call me sexist, but when I ...     sexism       1.0  
3  @g0ssipsquirrelx Wrong, ISIS follows the examp...     racism       1.0  
4                             #mkr No No No No No No       none       0.0  


In [6]:
#df['oh_label'] = df['Annotation'].map({0.0:"Not racism or sexism", 1.0:"racism or sexism"})
#df['labels'].fillna("No hate or offensive speech", inplace=True)
# Drop rows with NaN values in the 'labels' column
#df.dropna(subset=['labels'], inplace=True)
print(df.head())

                   index                     id  \
0  5.74948705591165E+017  5.74948705591165E+017   
1  5.71917888690393E+017  5.71917888690393E+017   
2  3.90255841338601E+017  3.90255841338601E+017   
3  5.68208850655916E+017  5.68208850655916E+017   
4  5.75596338802373E+017  5.75596338802373E+017   

                                                Text Annotation  oh_label  
0  @halalflaws @biebervalue @greenlinerzjm I read...       none       0.0  
1  @ShreyaBafna3 Now you idiots claim that people...       none       0.0  
2  RT @Mooseoftorment Call me sexist, but when I ...     sexism       1.0  
3  @g0ssipsquirrelx Wrong, ISIS follows the examp...     racism       1.0  
4                             #mkr No No No No No No       none       0.0  


In [7]:
df = df[['Text', 'Annotation']]
print(df.head())

                                                Text Annotation
0  @halalflaws @biebervalue @greenlinerzjm I read...       none
1  @ShreyaBafna3 Now you idiots claim that people...       none
2  RT @Mooseoftorment Call me sexist, but when I ...     sexism
3  @g0ssipsquirrelx Wrong, ISIS follows the examp...     racism
4                             #mkr No No No No No No       none


In [8]:
def clean(text):
    
    #convert text to lower case
    text = str(text).lower()
    
    # Remove URLs, HTML tags, special characters, and digits
    text = re.sub(r'http\S+|www\S+|<.*?>|[^a-zA-Z\s]', '', text)
    
    #Remove squared brackets and their contents
    #text = re.sub('\[.*?\]', '', text)
    
    #Remove URLs (http/https) and website links (www)
    #text = re.sub('https?://\S+|www\.\S+', '', text)
    
    #Remove HTML tags
    #text = re.sub('<.*?>+', '', text)
    #Remove percentage sign
    #text = re.sub('[%]','',text)
  
    #text = re.sub('\n', '', text)
    
    #Remove words containing digits
    #text = re.sub('\w*\d\w*', '', text)
    
    # Remove exclamation points, "@" symbols, and colons
    #text = re.sub(r'[!@:]', '', text)
    
    #Tokenize the text and remove stopwords
    #text = [word for word in text.split(' ') if word not in stopword]
    
    # Tokenize and remove stopwords
    tokens = [word for word in text.split() if word not in stopword]
    
    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    #Join the cleaned words back into a string
    #text = " ".join(text)
    
    #text = [stemmer.stem(word) for word in text.split(' ')]
    
    #text = " ".join(text)
    
    #return text
    return ' '.join(tokens)

df["Text"] = df["Text"].apply(clean)
print(df.head())

                                                Text Annotation
0  halalflaws biebervalue greenlinerzjm read cont...       none
1  shreyabafna idiot claim people tried stop beco...       none
2  rt mooseoftorment call sexist go auto place id...     sexism
3  gssipsquirrelx wrong isi follows example moham...     racism
4                                                mkr       none


In [9]:
x = np.array(df["Text"])
y = np.array(df["Annotation"])

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state= 42)

#cv = CountVectorizer()
# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)  # Limit features to top 5000
#x = cv.fit_transform(x)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


# Initialize Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier
clf.fit(X_train_tfidf, y_train)

#clf = DecisionTreeClassifier()

#clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [10]:
#Make predictions on the testing data
y_pred = clf.predict(X_test_tfidf)
#y_pred = model.predict_classes(X_test_pad)

In [11]:
#Evaluate the performance of the classifier
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy: ", accuracy*100)


Accuracy:  84.68842729970326


In [12]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

        none       0.86      0.92      0.89      2306
      racism       0.74      0.72      0.73       392
      sexism       0.85      0.65      0.74       672

    accuracy                           0.85      3370
   macro avg       0.82      0.77      0.79      3370
weighted avg       0.85      0.85      0.84      3370



In [13]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[2133   97   76]
 [ 107  281    4]
 [ 232    0  440]]


In [14]:
test_data = "I'm not sexist but women should not be working"

# Apply the same preprocessing steps to the test data
clean_test_data = clean(test_data)

# Transform the preprocessed test data using the same vectorizer used for training data
test_data_vectorized = vectorizer.transform([clean_test_data]).toarray()

# Make predictions on the test data
predicted_label = clf.predict(test_data_vectorized)





In [15]:
# Check if the predicted label is "racism" or "sexism"
if predicted_label in ["racism", "sexism"]:
    print("Cyberbullying")
else:
    print(predicted_label)

Cyberbullying
